# client

> Thin wrapper around an OpenAI-compatible chat completions API. Knows nothing about notebooks or JupyterLab.

In [ ]:
#| default_exp client

In [ ]:
#| hide
from nbdev.showdoc import *

## Model listing

Different providers return different fields from `/models`: OpenRouter has `name` (`"Meta: Llama 3.1 8B"`), OpenAI has `owned_by`, Anthropic's OpenAI-compatible endpoint has `display_name`. `model_entry` tolerates all of them.

In [ ]:
#| export
def model_entry(model # A model object as returned by `client.models.list()`
               )->dict: # dict with keys company/name/id
    """Builds yasi's model dict from an API model object, tolerating provider-specific fields
    (OpenRouter: `name`, OpenAI: `owned_by`, Anthropic: `display_name`)"""
    name = getattr(model, 'name', None)
    display_name = getattr(model, 'display_name', None)
    owned_by = getattr(model, 'owned_by', None)
    if name: company = name.split(':')[0]
    elif owned_by: company = owned_by
    elif '/' in model.id: company = model.id.split('/')[0]
    else: company = model.id.split('-')[0]
    return {"company": company,
            "name": name if name else (display_name if display_name else model.id.split('/')[-1]),
            "id": model.id}

In [ ]:
from types import SimpleNamespace

# OpenRouter style
m = SimpleNamespace(id='meta-llama/llama-3.1-8b-instruct', name='Meta: Llama 3.1 8B Instruct')
assert model_entry(m) == {'company': 'Meta', 'name': 'Meta: Llama 3.1 8B Instruct', 'id': 'meta-llama/llama-3.1-8b-instruct'}

# OpenAI style
m = SimpleNamespace(id='gpt-4o-mini', owned_by='openai', name=None)
assert model_entry(m) == {'company': 'openai', 'name': 'gpt-4o-mini', 'id': 'gpt-4o-mini'}

# Anthropic (OpenAI-compatible endpoint) style
m = SimpleNamespace(id='claude-sonnet-5', display_name='Claude Sonnet 5')
assert model_entry(m) == {'company': 'claude', 'name': 'Claude Sonnet 5', 'id': 'claude-sonnet-5'}

## Provider quirks

Anthropic's OpenAI-compatible endpoint only accepts `temperature` between 0 and 1 (yasi's slider goes to 2), requires `max_tokens`, and ignores `presence_penalty`. `prepare_params` adjusts the request parameters accordingly, based on the base url.

In [ ]:
#| export
ANTHROPIC_DEFAULT_MAX_TOKENS = 4096

def prepare_params(base_url, # base url of the api, used to detect the provider
                   model:str=None,
                   max_tokens:int=None,
                   temperature:float=None,
                   presence_penalty:float=None
                  )->dict: # keyword arguments for `chat.completions.create`
    """Adjusts request parameters to provider quirks (currently: Anthropic's temperature range,
    required max_tokens and unsupported presence_penalty)"""
    if 'anthropic' in str(base_url or ''):
        if temperature is not None: temperature = max(0.0, min(temperature, 1.0))
        if max_tokens is None: max_tokens = ANTHROPIC_DEFAULT_MAX_TOKENS
        presence_penalty = None
    return dict(model=model, max_tokens=max_tokens,
                temperature=temperature, presence_penalty=presence_penalty)

In [ ]:
# Anthropic: clamp temperature, default max_tokens, drop presence_penalty
p = prepare_params('https://api.anthropic.com/v1/', model='claude-sonnet-5',
                   temperature=1.7, presence_penalty=0.5)
assert p == {'model': 'claude-sonnet-5', 'max_tokens': 4096, 'temperature': 1.0, 'presence_penalty': None}

# other providers: untouched
p = prepare_params('https://openrouter.ai/api/v1', model='meta-llama/llama-3.1-8b-instruct',
                   temperature=1.7, presence_penalty=0.5)
assert p == {'model': 'meta-llama/llama-3.1-8b-instruct', 'max_tokens': None,
             'temperature': 1.7, 'presence_penalty': 0.5}

In [ ]:
#| export
import openai

class ChatClient:
    """Wraps an `openai.Client` for listing models and sending chat completions."""
    def __init__(self,
                 api_key : str=None,  # api key for the openai api
                 openai_base_url : str=None # base url of the openai api
                ):
        self.client = openai.Client(base_url=openai_base_url, api_key=api_key)
        self.latest_response = None

    def get_models(self):
        """Create list of model objects from the Openai client"""
        return [model_entry(model) for model in self.client.models.list()]

    def chat(self,
             messages: list, # A list of messages comprising the conversation so far.
             model: str=None, # model id for the openai api
             max_tokens: int=None,
             temperature: float=None,
             presence_penalty: float=None
            )->str: # The response text, or an error description
        """Send messages a.k.a dialog to the Openai chat completions API"""
        params = prepare_params(self.client.base_url,
                                model=model,
                                max_tokens=max_tokens,
                                temperature=temperature,
                                presence_penalty=presence_penalty)
        try:
            response = self.client.chat.completions.create(messages=messages, **params)
            self.latest_response = response
            response_text = response.choices[0].message.content.strip()
        except Exception as e:
            response_text = f'There was an problem communicating with the Openai API:\n\n{e}'
        return response_text

Usage against OpenRouter (needs a network connection and an api key, so this cell is not executed during tests):

In [ ]:
#| notest
cc = ChatClient(openai_base_url="https://openrouter.ai/api/v1", api_key=None)
models = cc.get_models()
models[:3]

# Anthropic's OpenAI-compatible endpoint (note the trailing /v1/):
# cc = ChatClient(openai_base_url="https://api.anthropic.com/v1/", api_key=os.environ["ANTHROPIC_API_KEY"])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()